# سبق 03 - ایجنٹک ڈیزائن پیٹرنز

اس سبق میں، ہم مؤثر AI ایجنٹس بنانے کے لیے تین بنیادی ڈیزائن پیٹرنز کا جائزہ لیں گے:

1. **واضح ایجنٹ ہدایات** — ایسے درست، کردار متعین کرنے والے پرامپٹس تیار کرنا جو ایجنٹ کے رویے کی رہنمائی کریں  
2. **پائڈینٹک ماڈلز کے ساتھ ساختہ آؤٹ پٹ** — یہ یقینی بنانا کہ ایجنٹس پیش گوئی شدہ، تصدیق شدہ ڈیٹا واپس کریں  
3. **سنگل ریسپونسبلیٹی ایجنٹس** — ایسے مرکزی ایجنٹس ڈیزائن کرنا جو ہر ایک کام کو بخوبی انجام دیں  

ہم ہر پیٹرن کو ایک **سفر کی منزل کی سفارش کنندہ** منظر نامے پر لاگو کریں گے، بتدریج ایسا نظام تیار کرتے ہوئے جو منزلیں تجویز کر سکے، دستیابی چیک کر سکے، اور لاجسٹکس سنبھال سکے۔


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## نمونہ 1: ایجنٹ کے لیے واضح ہدایات

سب سے مؤثر نمونہ سب سے آسان بھی ہے: اپنے ایجنٹ کے لیے واضح، تفصیلی ہدایات لکھنا۔

اچھی ہدایات کی تعریف ہوتی ہے:
- **کون** ایجنٹ ہے (کردار اور لہجہ)
- **کیا** اسے کرنا چاہیے (قدم بہ قدم ذمہ داریاں)
- **کیسے** اسے برتاؤ کرنا چاہیے (پابندیاں اور انداز)

نیچے، ہم ایک سفری کونسیئرج ایجنٹ بناتے ہیں جس کی واضح ہدایات ہوتی ہیں جو ہر جواب کو تشکیل دیتی ہیں جو وہ دیتا ہے۔


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: پائیڈینٹک ماڈلز کے ساتھ منظم آؤٹ پٹ

آزاد فارم متن گفتگو کے لیے مفید ہے، لیکن نیچے والے نظاموں کو منظم ڈیٹا کی ضرورت ہوتی ہے۔
**پائیڈینٹک ماڈلز** کو ایک **ٹول فنکشن** کے ساتھ جوڑ کر، ہم کر سکتے ہیں:

- ایجنٹ کے آؤٹ پٹ کے لیے ایک درست اسکیمہ کی تعریف
- جوابات کو خودکار طور پر درست قرار دینا
- ایجنٹ کے نتائج کو ایپلیکیشن لاجک میں قابل بھروسہ طریقے سے شامل کرنا

ہم ایک ایسا ٹول بھی متعارف کراتے ہیں جو منزل کی تفصیلات لوٹاتا ہے تاکہ ایجنٹ اپنی سفارشات کو حقیقی ڈیٹا کی بنیاد پر رکھ سکے۔


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## پیٹرن 3: واحد ذمہ داری والے ایجنٹس

پیچیدہ کاموں میں کام کو متعدد مخصوص ایجنٹس کے درمیان تقسیم کرنے سے فائدہ ہوتا ہے، ہر ایک کی ایک واحد ذمہ داری ہوتی ہے:

- ایک **مقصد کے ماہر** جو مقامات اور دستیابی کے بارے میں جانتا ہو
- ایک **لاجسٹکس پلانر** جو پروازوں، ہوٹلوں، اور سفرناموں کو سنبھالتا ہو

یہ سافٹ ویئر انجینئرنگ کے اصول *علاقوں کی تقسیم* کی عکاسی کرتا ہے — ہر ایجنٹ کو خود مختاری سے آزمانا، برقرار رکھنا، اور بہتر بنانا آسان ہوتا ہے۔


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## خلاصہ

اس سبق میں ہم نے تین ایجنٹک ڈیزائن پیٹرنز کو ایک سفری سفارش کنندہ منظر نامے پر لاگو کیا:

| پیٹرن | کلیدی خیال | فائدہ |
|---|---|---|
| **واضح ہدایات** | پرسونا، ذمہ داریاں، اور حدود کو پہلے سے متعین کریں | مستقل، برانڈ کے مطابق ایجنٹ کا رویہ |
| **منظم آؤٹ پٹ** | ردعمل کے فارمیٹ کے طور پر Pydantic ماڈلز کا استعمال کریں | تصدیق شدہ، مشین سے قابلِ پڑھائی نتائج |
| **واحد ذمہ داری** | ہر ایجنٹ کو ایک مرکزیت والی ذمہ داری دیں | ٹیسٹ کرنا، برقرار رکھنا، اور جوڑنا آسان |

یہ پیٹرنز قدرتی طور پر مل کر کام کرتے ہیں — آپ واضح ہدایات کو منظم آؤٹ پٹ کے ساتھ ایک واحد ذمہ داری والے ایجنٹ میں جوڑ کر مضبوط، پروڈکشن کے لیے تیار نظام بنا سکتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دستخطی اعلان**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کی کوشش کرتے ہیں، براہ کرم اس بات کا خیال رکھیں کہ خودکار تراجم میں غلطیاں یا کمی بیشی ہو سکتی ہے۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ سمجھا جائے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمعے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم پر عائد نہیں ہوتی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
